# Masked diffusion language model (`variant: diffusion`)

<img src="../../assets/explainers/diffusion.gif" width="100%">

**Training**: raising the noise level $t$ masks more and more characters (`?`), then the bidirectional denoiser guesses every mask at once (green = right, red = wrong). **Generation**: start from 40 masks at $t=1$; every step the model guesses *every* `?` (faint gold letters, brighter = more sure), keeps a few of them (they pop) and re-masks the rest, while $t$ walks to 0. 16 forward passes for 40 characters.

## Intuition

Instead of writing left to right, start from a blank canvas of `[MASK]` tokens and fill it in over a few
rounds, any order, many positions per round. Training mirrors this: hide a random fraction of a real
sentence and learn to restore it (MDLM, Sahoo et al. 2024; absorbing-state D3PM).

## Math

**Forward (corruption)** with the log-linear schedule $\alpha_t = 1 - t$:
$$q(x_t^i \mid x_0^i) = (1-t)\,\delta_{x_0^i} + t\,\delta_{[\mathrm{M}]}.$$

**Objective** (continuous-time ELBO): with $t \sim U[\epsilon, 1]$,
$$\mathcal{L} = \mathbb{E}_{t, x_t}\Big[\frac{1}{t}\sum_{i:\,x_t^i = [\mathrm{M}]} -\log p_\theta(x_0^i \mid x_t, t)\Big].$$

**Reverse step** $t \to s < t$: a masked token stays masked with probability $s/t$, otherwise it becomes
a sample from $p_\theta(x_0^i \mid x_t)$ - so each masked position is revealed w.p. $(t-s)/t$; the last
step reveals everything.

**The catch.** Tokens revealed in the *same* step are sampled independently given $x_t$. With few steps
the model can mix branches ("rodel" = "robot" ⊕ "model"); more steps → fewer simultaneous reveals →
closer to the true joint.

## In the code

| File | What |
|---|---|
| `models_diffusion/model.py` | `MaskedDiffusionLM`, `_corrupt`, `diffusion_loss`, `sample_diffusion` |
| `models_diffusion/noise_schedule.py` | `LogLinearSchedule` ($\alpha_t = 1-t$) |
| `models_diffusion/metrics.py` | `masked_accuracy` |

In [ ]:
import sys
from pathlib import Path

# Run from anywhere inside the repo: put src/ on the path if hale_llm is not installed.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from IPython.display import HTML, display

import hale_llm.models  # noqa: F401  - registers every paradigm
from hale_llm.core.registry import get_loss, get_sampler, get_variant
from hale_llm.dataloader.collate import get_collate
from hale_llm.visualization.explainers import CORPUS, get_explainer

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
print("training corpus:")
for sentence in CORPUS:
    print("  ", repr(sentence))

## Train a toy model

Bidirectional attention + time conditioning (`use_time_cond: true`). The explainer's recipe trains a ~100k-parameter model on the four corpus sentences
in a few seconds on a CPU - small enough to inspect every number, trained enough to show real
behaviour.

In [ ]:
explainer = get_explainer("diffusion")
run = explainer.train()
tok, model, cfg = run.tokenizer, run.model, run.cfg
print(type(model).__name__, f"{sum(p.numel() for p in model.parameters()):,} parameters")
print("config:", {k: v for k, v in cfg.model.model_dump().items()
                  if k in ("arch", "d_model", "n_layers", "attn_type", "use_time_cond",
                           "block_size", "n_mtp_heads", "fm_schedule")})

fig, ax = plt.subplots(figsize=(6.5, 2.4))
ax.semilogy(run.losses, color="#264653")
ax.set(xlabel="step", ylabel="training loss", title="toy training")
plt.show()

## The forward process

In [ ]:
from hale_llm.models.models_diffusion.model import _corrupt

for t in (0.1, 0.4, 0.7, 0.95):
    torch.manual_seed(0)
    x_t, masked = _corrupt(run.ids, torch.tensor([t]), tok.mask_token_id)
    print(f"t={t:<4}", "".join("_" if m else c for c, m in zip(run.text, masked[0].tolist())),
          f"  ({int(masked.sum())}/48 masked, loss weight 1/t = {1 / t:.1f})")

## What the denoiser sees and predicts

In [ ]:
torch.manual_seed(3)
x_t, masked = _corrupt(run.ids, torch.tensor([0.6]), tok.mask_token_id)
with torch.no_grad():
    probs = F.softmax(model({"x_t": x_t, "t": torch.tensor([0.6])})[0], -1)
guess = "".join(tok.token(int(i)) if m else "·" for i, m in zip(probs.argmax(-1), masked[0]))
print("x_t   :", "".join("_" if m else c for c, m in zip(run.text, masked[0].tolist())))
print("guess :", guess)
print("truth :", run.text)
print("masked accuracy:", float((probs.argmax(-1) == run.ids[0])[masked[0]].float().mean()))

## Sampling budget vs coherence

Fewer steps = more tokens revealed per step = more independence errors. "valid" = the output is exactly a
training sentence.

In [ ]:
def validity(steps_list, seeds=24, temperature=1.0):
    """Share of samples that are exactly one of the training sentences, per sampling-step budget."""
    prompt = run.ids[:, :8]
    sampler = get_sampler(cfg.variant)
    out = {}
    for steps in steps_list:
        ok = 0
        for seed in range(seeds):
            torch.manual_seed(seed)
            ids = sampler(model, tok, prompt, max_new_tokens=40, temperature=temperature, sampling_steps=steps)
            ok += tok.decode(ids[0]) in CORPUS
        out[steps] = ok / seeds
    return out

In [ ]:
scores = validity([2, 4, 8, 16, 40])
plt.figure(figsize=(5, 2.6)); plt.plot(list(scores), list(scores.values()), "o-", color="#264653")
plt.xscale("log"); plt.xlabel("sampling steps"); plt.ylabel("valid sentences"); plt.ylim(-0.05, 1.05)
plt.title("parallel decoding error vs steps"); plt.show()
print(scores)

## The traced generation

In [ ]:
trace = explainer.trace(run, seed=0)
assert torch.equal(trace.ids, explainer.reference(run, seed=0))
for i, s in enumerate(trace.steps):
    canvas = "".join("_" if v == tok.mask_token_id else tok.token(v) for v in s.x)
    print(f"t {s.t_from:.2f}→{s.t_to:.2f}  +{len(s.revealed):2d}  {canvas}")

In [ ]:
# Re-render the animation from this notebook's model. The committed copy shown at the top is
# assets/explainers/diffusion.gif (regenerate all of them with `hale-llm-explain`).
from PIL import Image

out = ROOT / "notebooks" / "models" / "outputs" / "diffusion.gif"  # git-ignored
explainer.build(out, run=run)
with Image.open(out) as gif:
    print(out.relative_to(ROOT), f"{gif.n_frames} frames, {gif.size[0]}x{gif.size[1]},",
          f"{out.stat().st_size / 1024:.0f} KB")
# In Jupyter: display(HTML('<img src="outputs/diffusion.gif" width="100%">'))

## Takeaways

* Any-order, parallel generation: the number of forward passes is the number of steps, not tokens.
* The loss is an ELBO (1/t-weighted masked CE), so likelihoods are bounds, not exact.
* Fixed canvas length; tokens revealed together are conditionally independent - the "rodel" failure.
* Fixed in this repo while building the animation: the sampler's last step used to reveal with
  probability $(t-\epsilon)/t < 1$ and could leave masks in the output; it now reveals everything.

**Exercises.** (1) Sample with `temperature=0.3` and re-run `validity`. (2) Swap `arch: lgt` / `dit`
via `train_toy("diffusion", CORPUS, model={...})`.